# `synthetic_data`

Synthetic stand-ins for the Kaggle datasets.

PURPOSE: let you smoke-test the whole pipeline without downloading anything.
The numbers produced from this data are NOT scientific results - never report them in the paper.
Real data: see README.md (Kaggle 'Crop Production in India' + 'Crop Recommendation Dataset').

> Part of the crop yield & recommendation project. Imported by `run_all.py` (via the notebook loader in `src/__init__.py`) or runnable directly here.

In [ ]:
# Setup: make the project root importable when this notebook is opened directly in Jupyter
import sys
from pathlib import Path
_root = Path.cwd().resolve()
if _root.name == "src":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import numpy as np
import pandas as pd
from src import config as C

STATES = ["Uttar Pradesh", "Punjab", "Maharashtra", "Karnataka", "Tamil Nadu", "West Bengal",
          "Bihar", "Gujarat", "Rajasthan", "Madhya Pradesh", "Andhra Pradesh", "Haryana"]

# crop: (base yield t/ha, seasons, rainfall sensitivity)
CROPS = {
    "Rice": (2.6, ["Kharif", "Rabi", "Summer"], 1.0), "Wheat": (2.9, ["Rabi"], 0.3),
    "Maize": (2.7, ["Kharif", "Rabi"], 0.6), "Sugarcane": (65.0, ["Whole Year"], 0.7),
    "Cotton(lint)": (0.5, ["Kharif"], 0.8), "Groundnut": (1.2, ["Kharif", "Rabi"], 0.7),
    "Soyabean": (1.0, ["Kharif"], 0.8), "Potato": (20.0, ["Rabi"], 0.2),
    "Onion": (15.0, ["Rabi", "Kharif"], 0.3), "Banana": (30.0, ["Whole Year"], 0.5),
    "Arhar/Tur": (0.8, ["Kharif"], 0.6), "Gram": (0.9, ["Rabi"], 0.2),
    "Bajra": (1.1, ["Kharif"], 0.5), "Jowar": (1.0, ["Kharif", "Rabi"], 0.5),
    "Ragi": (1.4, ["Kharif"], 0.6), "Rapeseed &Mustard": (1.0, ["Rabi"], 0.2),
}

In [ ]:
def make_yield_data(districts_per_state=10, seed=C.SEED, years=range(1997, 2016)):
    rng = np.random.default_rng(seed)
    state_f = {s: rng.uniform(0.75, 1.3) for s in STATES}
    state_rain = {s: rng.uniform(500, 1600) for s in STATES}
    rain_rows, rain = [], {}
    for s in STATES:
        for y in years:
            rain[(s, y)] = max(150, rng.normal(state_rain[s], 130))
            rain_rows.append((s, y, round(rain[(s, y)], 1)))
    pd.DataFrame(rain_rows, columns=["State", "Year", "rainfall_mm"]).to_csv(C.RAIN_RAW, index=False)

    rows = []
    crop_names = list(CROPS)
    for s in STATES:
        for d in range(districts_per_state):
            dname = f"{s.split()[0][:4].upper()}_DIST{d+1}"
            dist_f = rng.uniform(0.8, 1.2)
            for crop in rng.choice(crop_names, size=rng.integers(6, 11), replace=False):
                base, seasons, sens = CROPS[crop]
                for season in seasons:
                    for y in years:
                        if rng.random() < 0.08:
                            continue
                        r = rain[(s, y)]
                        rain_eff = np.exp(sens * 0.0007 * (r - state_rain[s]))
                        yld = (base * state_f[s] * dist_f * (1 + 0.015 * (y - 1997)) *
                               rain_eff * rng.lognormal(0, 0.12))
                        area = float(rng.lognormal(7.3, 1.0))
                        prod = area * yld
                        rows.append([s, dname, y, season, crop, round(area, 1),
                                     round(prod, 1) if rng.random() > 0.01 else np.nan])
    df = pd.DataFrame(rows, columns=["State_Name", "District_Name", "Crop_Year", "Season",
                                     "Crop", "Area", "Production"])
    # mimic Kaggle formatting quirks (trailing spaces) so the cleaning code is exercised
    df["Season"] = df["Season"].map(lambda x: x.ljust(11))
    df.to_csv(C.YIELD_RAW, index=False)
    return df

In [ ]:
# crop: N, P, K, temperature, humidity, ph, rainfall  (approximate class centres)
PROFILES = {
    "rice": (80, 48, 40, 23.7, 82, 6.4, 236), "maize": (78, 48, 20, 22.4, 65, 6.2, 85),
    "chickpea": (40, 68, 80, 18.9, 16, 7.3, 80), "kidneybeans": (21, 68, 20, 20.0, 22, 5.7, 106),
    "pigeonpeas": (21, 68, 20, 27.7, 49, 5.8, 149), "mothbeans": (21, 48, 20, 28.2, 53, 6.8, 51),
    "mungbean": (21, 48, 20, 28.5, 85, 6.7, 48), "blackgram": (40, 67, 19, 29.9, 65, 7.0, 68),
    "lentil": (19, 68, 19, 24.5, 64, 6.9, 46), "pomegranate": (19, 19, 40, 21.8, 90, 6.4, 108),
    "banana": (100, 82, 50, 27.4, 80, 6.0, 104), "mango": (20, 27, 30, 31.2, 50, 5.8, 95),
    "grapes": (23, 133, 200, 23.8, 82, 6.0, 70), "watermelon": (99, 17, 50, 25.6, 85, 6.5, 50),
    "muskmelon": (100, 17, 50, 28.7, 92, 6.4, 25), "apple": (21, 134, 200, 22.6, 92, 5.9, 113),
    "orange": (20, 10, 10, 22.8, 92, 7.0, 110), "papaya": (50, 59, 50, 33.7, 92, 6.7, 143),
    "coconut": (22, 17, 31, 27.4, 95, 6.0, 176), "cotton": (118, 46, 20, 24.0, 80, 6.9, 80),
    "jute": (78, 46, 40, 25.0, 80, 6.7, 175), "coffee": (101, 29, 30, 25.5, 58, 6.8, 158),
}

In [ ]:
def make_reco_data(n_per_class=100, seed=C.SEED):
    rng = np.random.default_rng(seed)
    rows = []
    for crop, (n, p, k, t, h, ph, r) in PROFILES.items():
        sd = [0.12 * n + 4, 0.12 * p + 4, 0.12 * k + 4, 2.5, 6.0, 0.4, 0.15 * r + 6]
        mu = [n, p, k, t, h, ph, r]
        X = rng.normal(mu, sd, size=(n_per_class, 7))
        X[:, :3] = np.clip(X[:, :3], 0, 205)
        X[:, 4] = np.clip(X[:, 4], 14, 100)
        X[:, 5] = np.clip(X[:, 5], 3.5, 9.9)
        X[:, 6] = np.clip(X[:, 6], 20, 300)
        for row in X:
            rows.append([*np.round(row, 3), crop])
    df = pd.DataFrame(rows, columns=["N", "P", "K", "temperature", "humidity", "ph", "rainfall", "label"])
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    df.to_csv(C.RECO_RAW, index=False)
    return df

In [ ]:
def make_all():
    print("Generating SYNTHETIC data (pipeline smoke-test only)...")
    y = make_yield_data()
    r = make_reco_data()
    print(f"  yield rows: {len(y):,} | recommendation rows: {len(r):,}")

In [ ]:
if __name__ == "__main__":
    make_all()